In [1]:
import os
import cv2
import os
import numpy as np
import torch
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '../../')))
import torch
from gestures.network.models.basic_model import BasicModel
from gestures.network.models.super_resolution.safmn import SAFMN
import torch
import time
from gestures.network.models.sr_classifier.SRCnnTinyRadar import CombinedSRDrlnClassifier , MultiSRClassifier


In [3]:
from fvcore.nn import FlopCountAnalysis
import torchprofile
from ptflops import get_model_complexity_info
def get_model_params(model,model_name ,dummy_input):
    model.eval()
    print(dummy_input.shape)
    # Measure the time
    start_time = time.time()
    with torch.no_grad():  # Disable gradient calculation for inference
        _ = model(dummy_input)
    end_time = time.time()

    inference_time = end_time - start_time
    print(model_name)
    print(f"Inference Time: {inference_time:.6f} seconds")
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters: {num_params:,}")
    flops = torchprofile.profile_macs(model, args=(dummy_input,))
    print(f"Total FLOPs: {flops / 1e9:.2f} GFLOPs")


In [32]:
safmn = SAFMN()
data = torch.randn(1,2, 32, 492)
get_model_params(safmn,"SAFMN",data)

torch.Size([1, 2, 32, 492])
SAFMN
Inference Time: 0.245951 seconds
Total number of parameters: 233,996
Total FLOPs: 3.63 GFLOPs


/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::pow". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::sqrt". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::unsqueeze". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::gelu". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWa

In [6]:
from gestures.network.models.classifiers.tiny_radar import TinyRadarNN


tiny = TinyRadarNN()
dummy_input = torch.randn(5,1,2, 32, 492)
get_model_params(tiny,"TinyRadarNN",dummy_input)


torch.Size([5, 1, 2, 32, 492])
TinyRadarNN
Inference Time: 0.010926 seconds
Total number of parameters: 45,948
Total FLOPs: 0.08 GFLOPs


/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::permute". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(


In [4]:
from gestures.utils_processing_data import *
from gestures.data_loader2.dataset_factory import *
from gestures.setup import get_pc_cgf

files = os.listdir("/Users/netanelblumenfeld/Downloads/11G/tt")
gestures = [
    "PinchIndex",
    "PinchPinky",
    "FingerSlider",
    "FingerRub",
    "SlowSwipeRL",
    "FastSwipeRL",
    "Push",
    "Pull",
    "PalmTilt",
    "Circle",
    "PalmHold",
    "NoHand",
]
base_dir = "/Users/netanelblumenfeld/Downloads/11G/tt"
task = "sr_classifier"  # task = ["sr", "classifier", "sr_classifier"]
pc, data_dir, output_dir, device = get_pc_cgf()


batch_size = 1
dx, dy = 4,4
pre_processing_funcs = {
    "classifier": torch.nn.Sequential(
        ToTensor(),
        DownSampleOneSample(dx=dx, dy=dy, original_dims=False),
        NormalizeOneSample(),
        DopplerMapOneSample(),
    ),
    "sr_classifier": {
        "hr": torch.nn.Sequential(
            ToTensor(), NormalizeOneSample(), ComplexToRealOneSample()
        ),
        "lr": torch.nn.Sequential(
            ToTensor(),
            DownSampleOneSample(dx=dx, dy=dy, original_dims=False),
            NormalizeOneSample(),
            ComplexToRealOneSample(),
        ),
    },
    "sr": {
        "hr": torch.nn.Sequential(
            ToTensor(), NormalizeOneSample(), ComplexToRealOneSample()
        ),
        "lr": torch.nn.Sequential(
            ToTensor(),
            DownSampleOneSample(dx=dx, dy=dy, original_dims=False),
            NormalizeOneSample(),
            ComplexToRealOneSample(),
        ),
    },

}

data_loader = get_data_loader(
    task, batch_size, gestures, data_dir, pre_processing_funcs[task]
)
for x,y in data_loader['val']:
    lr_imgs = x
    hr_imgs = y
    break



In [57]:
safmn = SAFMN(upscaling_factor=4)

comb = CombinedSRDrlnClassifier(safmn, tiny)
dummy_input = comb.reshape_to_model_output(lr_imgs, hr_imgs, torch.device("cpu"))
get_model_params(comb,"SuperGestNet",dummy_input[0])



torch.Size([5, 1, 2, 2, 8, 123])
SuperGestNet
Inference Time: 0.109882 seconds
Total number of parameters: 279,962
Total FLOPs: 2.35 GFLOPs


/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::reshape". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::pow". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::sqrt". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::unsqueeze". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: Use

In [59]:
safmn = SAFMN(upscaling_factor=2)
comb = CombinedSRDrlnClassifier(safmn, tiny)
dummy_input = comb.reshape_to_model_output(lr_imgs, hr_imgs, torch.device("cpu"))
get_model_params(comb,"SuperGestNet",dummy_input[0])



torch.Size([5, 1, 2, 2, 16, 246])
SuperGestNet
Inference Time: 0.252208 seconds
Total number of parameters: 272,162
Total FLOPs: 8.85 GFLOPs


/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::reshape". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::pow". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::sqrt". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::unsqueeze". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/opt/homebrew/Caskroom/miniconda/base/envs/radar/lib/python3.12/site-packages/torchprofile/profile.py:22: Use

In [10]:
safmn2 = SAFMN(upscaling_factor=2)
safmn3 = SAFMN(upscaling_factor=3)
safmn4 = SAFMN(upscaling_factor=4)
tiny = TinyRadarNN()
multi = MultiSRClassifier(safmn2,safmn3,safmn4,tiny, torch.device("cpu"))
dummy_input = multi.reshape_to_model_output(lr_imgs, hr_imgs, torch.device("cpu"))
get_model_params(multi,"multi",dummy_input[0])


torch.Size([5, 1, 2, 2, 8, 123])
multi
Inference Time: 77.291309 seconds
Total number of parameters: 735,586


RuntimeError: shape '[5, 1, 2, 2, 32, 492]' is invalid for input of size 177120